# 18 — Fannie Mae multi-year data feasibility
## Stage 18B.2 — Compressed file benchmark
Check whether the existing quarterly ZIP can be scanned without extracting it. This stage only produces aggregate diagnostics; it does not sample loans or construct targets.

### 1. Locate the existing files

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, re, time, zipfile, ctypes
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW = ROOT / 'data/raw/Fannie_Mae'
ZIP = RAW / '2018Q1.zip'
CSV = RAW / '2018Q1/2018Q1.csv'
assert ZIP.is_file() and CSV.is_file(), 'Locate both existing 2018Q1 files.'
assert CSV.stat().st_size == 6928164686, 'CSV differs from Notebook 15.'
with zipfile.ZipFile(ZIP) as z:
    members = [x for x in z.infolist() if not x.is_dir() and Path(x.filename).name.lower() == '2018q1.csv']
    assert len(members) == 1, 'Review archive member names.'
    member = members[0]
    assert member.file_size == CSV.stat().st_size
    print(member.filename, member.file_size, 'uncompressed bytes')
print('ZIP size (GiB):', round(ZIP.stat().st_size / 2**30, 3))

2018Q1.csv 6928164686 uncompressed bytes
ZIP size (GiB): 0.444


### 2. Scan the ZIP and check the reviewed file layout
Count rows and unique contiguous loan histories, check origination dates, and calculate the hash of the decompressed bytes. Reading line by line avoids allocating a full quarterly DataFrame. Memory is reported as the Windows process lifetime peak working set, not incremental scan memory.

In [2]:
class MemoryCounters(ctypes.Structure):
    _fields_ = [('cb', ctypes.c_ulong), ('PageFaultCount', ctypes.c_ulong)] + [(name, ctypes.c_size_t) for name in ['PeakWorkingSetSize','WorkingSetSize','QuotaPeakPagedPoolUsage','QuotaPagedPoolUsage','QuotaPeakNonPagedPoolUsage','QuotaNonPagedPoolUsage','PagefileUsage','PeakPagefileUsage']]

def peak_ram_gib():
    try:
        p = MemoryCounters()
        p.cb = ctypes.sizeof(p)
        ctypes.windll.kernel32.GetCurrentProcess.restype = ctypes.c_void_p
        handle = ctypes.windll.kernel32.GetCurrentProcess()
        ok = ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.c_void_p(handle), ctypes.byref(p), p.cb)
        return round(p.PeakWorkingSetSize / 2**30, 4) if ok else None
    except AttributeError:
        return None

# Confirm the origination field position against the reviewed Notebook 15 code.
import ast
source_notebook = ROOT / 'notebooks/15_fannie_external_dataset_construction.ipynb'
assert source_notebook.is_file(), 'Notebook 15 is required to recover its reviewed field positions.'
positions = None
for cell in json.loads(source_notebook.read_text(encoding='utf-8'))['cells']:
    if cell['cell_type'] != 'code':
        continue
    try:
        tree = ast.parse(''.join(cell['source']))
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'POSITIONS' for t in node.targets):
            positions = ast.literal_eval(node.value)
assert isinstance(positions, dict) and 'orig' in positions, 'Review POSITIONS in Notebook 15.'
orig_index = positions['orig'] - 1
assert 0 <= orig_index < 113
print('Reviewed origination position:', orig_index + 1)

Reviewed origination position: 14


In [3]:
start = time.perf_counter()
sha = hashlib.sha256()
seen = set()
counts = Counter()
current = None
orig_dates = set()
rows = 0

def finish(key, dates):
    if key is None:
        return
    if key in seen:
        raise ValueError('Noncontiguous loan history; stop and review.')
    seen.add(key)
    if len(dates) != 1:
        counts['conflicting_or_missing'] += 1
        return
    date = next(iter(dates))
    if not re.fullmatch(rb'(0[1-9]|1[0-2])[0-9]{4}', date):
        counts['invalid_date'] += 1
        return
    counts[date[2:].decode()] += 1

with zipfile.ZipFile(ZIP) as z, z.open(member) as stream:
    for rows, raw in enumerate(stream, 1):
        sha.update(raw)
        parts = raw.rstrip(b'\r\n').split(b'|')
        if len(parts) != 113:
            raise ValueError(f'Unexpected field count at row {rows}')
        key = parts[1].strip()
        if not re.fullmatch(rb'[A-Za-z0-9]{1,12}', key):
            raise ValueError(f'Invalid loan ID at row {rows}')
        if key != current:
            finish(current, orig_dates)
            current, orig_dates = key, set()
        date = parts[orig_index].strip()
        if date:
            orig_dates.add(date)
        if rows % 2_000_000 == 0:
            print(f'{rows:,} rows scanned')
    finish(current, orig_dates)
scan_seconds = time.perf_counter() - start
scan_peak = peak_ram_gib()
assert rows == 23563399, 'Row count differs from Notebook 15.'
assert sum(counts.values()) == len(seen)
print('Rows:', rows, '| Unique loans:', len(seen), '| Scan seconds:', round(scan_seconds, 2))
display(pd.DataFrame(sorted(counts.items()), columns=['origination_year_or_status','loans']))

2,000,000 rows scanned
4,000,000 rows scanned
6,000,000 rows scanned
8,000,000 rows scanned
10,000,000 rows scanned
12,000,000 rows scanned
14,000,000 rows scanned
16,000,000 rows scanned
18,000,000 rows scanned
20,000,000 rows scanned
22,000,000 rows scanned
Rows: 23563399 | Unique loans: 458830 | Scan seconds: 192.37


,origination_year_or_status,loans
0,2013,1
1,2016,35
2,2017,197969
3,2018,260423
4,conflicting_or_missing,402


### 3. Verify the extracted CSV is identical
Hash the existing CSV in small blocks and compare it with the decompressed ZIP. No raw files are modified.

In [4]:
start = time.perf_counter()
csv_hash = hashlib.sha256()
with CSV.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        csv_hash.update(block)
hash_seconds = time.perf_counter() - start
assert csv_hash.hexdigest() == sha.hexdigest(), 'ZIP contents differ from the reviewed CSV.'
print('Exact byte-level match confirmed.')

Exact byte-level match confirmed.


### 4. Save aggregate benchmark results
Create a new review folder and ZIP. No individual loan identifiers or raw data are exported.

In [5]:
out = ROOT / 'data/processed/Fannie_Mae/notebook_18_processing_benchmark' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True, exist_ok=False)
summary = dict(zip_bytes=ZIP.stat().st_size, csv_bytes=CSV.stat().st_size, physical_rows=rows, unique_loans=len(seen), fields_per_row=113, origination_field_position=orig_index+1, zip_scan_seconds=round(scan_seconds, 2), csv_hash_seconds=round(hash_seconds, 2), process_peak_working_set_gib=scan_peak, decompressed_sha256=sha.hexdigest(), exact_csv_match=True, sampling_performed=False, targets_constructed=False)
(out / 'benchmark_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.DataFrame(sorted(counts.items()), columns=['origination_year_or_status','loans']).to_csv(out / 'origination_year_counts.csv', index=False)
review = out / 'notebook_18_stage_benchmark_review.zip'
with zipfile.ZipFile(review, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['benchmark_summary.json','origination_year_counts.csv']:
        z.write(out / name, name)
print(json.dumps(summary, indent=2))
print('Upload only:', review)

{
  "zip_bytes": 476739014,
  "csv_bytes": 6928164686,
  "physical_rows": 23563399,
  "unique_loans": 458830,
  "fields_per_row": 113,
  "origination_field_position": 14,
  "zip_scan_seconds": 192.37,
  "csv_hash_seconds": 56.29,
  "process_peak_working_set_gib": 0.1536,
  "decompressed_sha256": "b89043e7c2967d8f50bb3d7f5f694ee624a77ec86e9390a6333ee9ccf5ed3ebe",
  "exact_csv_match": true,
  "sampling_performed": false,
  "targets_constructed": false
}
Upload only: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_18_processing_benchmark\20261125T180308556377Z\notebook_18_stage_benchmark_review.zip


### Next review
Review scan time, memory, exact CSV reconciliation and origination-year counts before downloading additional files. This benchmark does not establish multi-year schema compatibility or sample sufficiency.